In [1]:
import pickle
import numpy as np
import tensorflow as tf
import random
from sklearn.metrics import accuracy_score, f1_score

with open('session_preprocessing.pkl', 'rb') as f:
    sv = pickle.load(f)

le             = sv['le']
scaler         = sv['scaler']
ecocrop        = sv['ecocrop']
X_train_scaled = sv['X_train_scaled']
X_val_scaled   = sv['X_val_scaled']
X_test_scaled  = sv['X_test_scaled']
y_train_oh     = sv['y_train_oh']
y_val_oh       = sv['y_val_oh']
y_test_oh      = sv['y_test_oh']

print(f"✅ Preprocessing dimuat")
print(f"   Train : {X_train_scaled.shape}")
print(f"   Val   : {X_val_scaled.shape}")
print(f"   Test  : {X_test_scaled.shape}")

✅ Preprocessing dimuat
   Train : (1540, 7)
   Val   : (330, 7)
   Test  : (330, 7)


In [2]:
# ============================================================
# SEED
# ============================================================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# ============================================================
# DATA LOSS
# ============================================================
def data_loss_fn(y_true, y_pred):
    y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0)
    return -tf.reduce_mean(
        tf.reduce_sum(y_true * tf.math.log(y_pred), axis=1)
    )

# ============================================================
# TRAINING LOOP
# ============================================================
def train_model(model, X_tr, y_tr, X_vl, y_vl,
                loss_fn, epochs=100, batch_size=32,
                lr=0.001, patience=10):

    optimizer = tf.keras.optimizers.Adam(learning_rate=lr)
    history   = {
        'train_loss': [], 'val_loss': [],
        'train_acc' : [], 'val_acc' : []
    }
    n = X_tr.shape[0]

    best_val_loss    = np.inf
    patience_counter = 0
    best_weights     = None
    best_epoch       = 0

    for epoch in range(epochs):
        np.random.seed(SEED + epoch)
        idx  = np.random.permutation(n)
        X_tr = X_tr[idx]
        y_tr = y_tr[idx]

        train_losses  = []
        train_correct = 0

        for i in range(0, n, batch_size):
            X_batch = tf.constant(X_tr[i:i+batch_size], dtype=tf.float32)
            y_batch = tf.constant(y_tr[i:i+batch_size], dtype=tf.float32)

            with tf.GradientTape() as tape:
                loss = loss_fn(model, X_batch, y_batch)

            grads = tape.gradient(loss, model.trainable_variables)
            optimizer.apply_gradients(zip(grads, model.trainable_variables))
            train_losses.append(loss.numpy())

            y_pred_batch  = model(X_batch, training=False)
            train_correct += np.sum(
                np.argmax(y_pred_batch.numpy(), axis=1) ==
                np.argmax(y_batch.numpy(), axis=1)
            )

        X_vl_tf  = tf.constant(X_vl, dtype=tf.float32)
        y_vl_tf  = tf.constant(y_vl, dtype=tf.float32)
        val_loss = loss_fn(model, X_vl_tf, y_vl_tf).numpy()
        y_pred_v = model(X_vl_tf, training=False)
        val_acc  = np.mean(
            np.argmax(y_pred_v.numpy(), axis=1) ==
            np.argmax(y_vl, axis=1)
        )
        train_acc = train_correct / n

        history['train_loss'].append(np.mean(train_losses))
        history['val_loss'].append(val_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)

        # Early stopping
        if val_loss < best_val_loss:
            best_val_loss    = val_loss
            patience_counter = 0
            best_weights     = model.get_weights()
            best_epoch       = epoch + 1
        else:
            patience_counter += 1
            if patience_counter >= patience:
                break

    if best_weights is not None:
        model.set_weights(best_weights)

    return history, best_epoch

print("✅ Loss function & training loop siap")

✅ Loss function & training loop siap


In [3]:
# ============================================================
# EKSPERIMEN ARSITEKTUR — BASELINE ONLY
# ============================================================

def build_mlp_custom(hidden_layers):
    layers = [tf.keras.layers.Input(shape=(7,))]
    for units in hidden_layers:
        layers.append(tf.keras.layers.Dense(units, activation='relu'))
    layers.append(tf.keras.layers.Dense(22, activation='softmax'))
    return tf.keras.Sequential(layers)

def baseline_loss_fn(model, X_batch, y_batch):
    y_pred = model(X_batch, training=True)
    return data_loss_fn(y_batch, y_pred)

# Arsitektur yang diuji
architectures = {
    'A: 7→8→22'    : [8],
    'B: 7→16→22'   : [16],
    'C: 7→32→22'   : [32],
    'D: 7→16→8→22' : [16, 8],
}

results_arch = {}

for arch_name, hidden in architectures.items():
    print(f"\n{'='*50}")
    print(f"  {arch_name}")
    print(f"{'='*50}")

    tf.random.set_seed(SEED)
    model_temp   = build_mlp_custom(hidden)
    total_params = model_temp.count_params()
    print(f"  Total params: {total_params}\n")

    history_temp, best_epoch_temp = train_model(
        model_temp,
        X_train_scaled, y_train_oh,
        X_val_scaled,   y_val_oh,
        baseline_loss_fn,
        epochs=100, patience=10
    )

    # Evaluasi test set
    X_test_tf = tf.constant(X_test_scaled, dtype=tf.float32)
    y_pred    = model_temp(X_test_tf, training=False).numpy()
    y_true    = np.argmax(y_test_oh, axis=1)
    y_pred_l  = np.argmax(y_pred,    axis=1)

    acc = accuracy_score(y_true, y_pred_l)
    f1  = f1_score(y_true, y_pred_l, 
                   average='macro', zero_division=0)

    results_arch[arch_name] = {
        'params'    : total_params,
        'best_epoch': best_epoch_temp,
        'best_val'  : max(history_temp['val_acc']),
        'test_acc'  : acc,
        'test_f1'   : f1
    }

    print(f"\n  ✅ Selesai")
    print(f"     Best Epoch : {best_epoch_temp}")
    print(f"     Best Val   : {max(history_temp['val_acc'])*100:.2f}%")
    print(f"     Test Acc   : {acc*100:.2f}%")
    print(f"     Test F1    : {f1*100:.2f}%")

# Ringkasan
print(f"\n{'='*65}")
print(f"  RINGKASAN")
print(f"{'='*65}")
print(f"{'Arsitektur':<20} {'Params':>8} {'Best Val':>10} "
      f"{'Test Acc':>10} {'Test F1':>10}")
print(f"{'-'*65}")
for arch_name, res in results_arch.items():
    print(f"{arch_name:<20} "
          f"{res['params']:>8} "
          f"{res['best_val']*100:>9.2f}% "
          f"{res['test_acc']*100:>9.2f}% "
          f"{res['test_f1']*100:>9.2f}%")


  A: 7→8→22
  Total params: 262


  ✅ Selesai
     Best Epoch : 100
     Best Val   : 97.27%
     Test Acc   : 97.27%
     Test F1    : 97.28%

  B: 7→16→22
  Total params: 502


  ✅ Selesai
     Best Epoch : 100
     Best Val   : 99.09%
     Test Acc   : 98.18%
     Test F1    : 98.18%

  C: 7→32→22
  Total params: 982


  ✅ Selesai
     Best Epoch : 100
     Best Val   : 99.70%
     Test Acc   : 98.79%
     Test F1    : 98.78%

  D: 7→16→8→22
  Total params: 462


  ✅ Selesai
     Best Epoch : 100
     Best Val   : 98.79%
     Test Acc   : 98.48%
     Test F1    : 98.48%

  RINGKASAN
Arsitektur             Params   Best Val   Test Acc    Test F1
-----------------------------------------------------------------
A: 7→8→22                 262     97.27%     97.27%     97.28%
B: 7→16→22                502     99.09%     98.18%     98.18%
C: 7→32→22                982     99.70%     98.79%     98.78%
D: 7→16→8→22              462     98.79%     98.48%     98.48%
